# V-Max BC pre-training on Colab (hanam/jeju, hard/easy pools)

Runs `algorithm=bc` training from `as-fast-as-anyone` (V-Max fork) on a Colab GPU instead of the local GTX 1660 Super (6GB VRAM / 15GB RAM).

**Data source**: the contest organizer's Drive folder, laid out as

```
Motion planning and prediction/train/
  hanam/<date>.tar.gz
  jeju/<date>.tar.gz
  livinglab/<date>.tar.gz     <- NOT used (the contest evaluates hanam/jeju)
```

Add that shared folder as a shortcut in your own Drive first (open the share -> "Add shortcut to Drive"), then fix `TRAIN_ROOT` below.

**What this notebook does**

1. `scripts/prepare_archives_91f.py` walks the per-date archives one at a time: extract -> convert to 91-step WOMD records (`make_91f`) -> score difficulty (`score_scenarios`) -> tar the result into a Drive cache -> delete the raw files. Peak local disk is one date, not the whole dataset, and a finished date is never converted twice (a killed session resumes by untarring the cache).
2. `scripts/split_hard_easy_pools.py --drop-frac 0.2 --hard-frac 0.4` throws away the bottom 20% (plain lane-keeping) per site and splits the rest into **4 pools**: `hanam_hard`, `hanam_easy`, `jeju_hard`, `jeju_easy`.
3. BC trains on all 4 as a weighted mixture, so the site ratio and the difficulty ratio are tuned independently.
4. Checkpoints go straight to Drive, and BC training resumes from the last one, so a disconnect mid-run costs one checkpoint interval.

Also upload the fixed 300-scenario **evaluation** set as a tar (so every model - local and Colab - is scored against the exact same set): locally, `tar -chf data/val_sample_shards_hanam.tar -C data/eval/val_sample_shards_hanam .` (~930MB, dereferenced so the symlinks survive), then upload it to `MyDrive/vmax_workdir/data/val_sample_shards_hanam.tar`.

Runtime > Change runtime type > pick a GPU. **Prefer an A100/L4 runtime if you have Colab Pro** - not for the GPU, for the vCPUs: the conversion step is pure CPU and a 2-vCPU T4 runtime converts roughly 6x slower.

In [ ]:
!nvidia-smi
!nproc && free -g && df -h /content

## 1. Mount Drive and check the layout

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Edit if the shared folder landed somewhere else.
TRAIN_ROOT = "/content/drive/MyDrive/Motion Planning and Prediction/train"
DRIVE_WORKDIR = "/content/drive/MyDrive/vmax_workdir"
SITES = "hanam,jeju"  # livinglab deliberately excluded
MAX_ARCHIVES_PER_SITE = 6  # dates per site, spread over its whole range; 0 for everything

os.environ["TRAIN_ROOT"] = TRAIN_ROOT
os.environ["DRIVE_WORKDIR"] = DRIVE_WORKDIR
os.environ["SITES"] = SITES
os.environ["MAX_ARCHIVES_PER_SITE"] = str(MAX_ARCHIVES_PER_SITE or 10**6)

assert os.path.isdir(TRAIN_ROOT), f"Missing {TRAIN_ROOT} - fix the path (did you Add shortcut to Drive?)."
for site in SITES.split(","):
    sdir = os.path.join(TRAIN_ROOT, site)
    assert os.path.isdir(sdir), f"Missing {sdir}"
    archives = sorted(f for f in os.listdir(sdir) if f.endswith((".tar.gz", ".tgz", ".tar")))
    print(f"{site}: {len(archives)} date archives, e.g. {archives[:3]}")

# Only needed for the checkpoint sweep in section 9 - training itself never reads it.
EVAL_TAR = f"{DRIVE_WORKDIR}/data/val_sample_shards_hanam.tar"
HAS_EVAL_TAR = os.path.exists(EVAL_TAR)
print("eval set:", "found" if HAS_EVAL_TAR else f"MISSING {EVAL_TAR} - sections 6/9 will skip it")

## 2. Clone the repo and set up the environment (uv, pinned by uv.lock)

In [ ]:
%cd /content
!rm -rf as-fast-as-anyone
!git clone https://github.com/gm2256/as-fast-as-anyone.git
%cd /content/as-fast-as-anyone/V-Max

!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"{os.path.expanduser('~')}/.local/bin:" + os.environ["PATH"]
!uv --version

In [ ]:
# Installs its own Python 3.12 (per .python-version) regardless of Colab's system Python,
# and resolves the exact versions pinned in uv.lock (same env as the local machine).
!uv sync

## 3. Smoke test: one date archive per site

Do not skip this. It validates the Drive path, the archive contents and the record schema in a few minutes, and - more importantly - it prints **seconds per file**, which is the number to multiply out before committing to the full run in step 4.

In [ ]:
!uv run python scripts/prepare_archives_91f.py "$TRAIN_ROOT" /content/data/smoke_91f \
    --sites "$SITES" \
    --scores-dir /content/data/smoke_scores \
    --windows 100 \
    --max-archives-per-site 1
!find /content/data/smoke_91f -name '*.tfrecord' | wc -l
!du -sh /content/data/smoke_91f

In [ ]:
# Extrapolate before the real run: per-date file count and size x the dates you plan to convert.
import os

n_files = sum(len(fs) for _, _, fs in os.walk("/content/data/smoke_91f"))
size_gb = sum(
    os.path.getsize(os.path.join(d, f))
    for d, _, fs in os.walk("/content/data/smoke_91f") for f in fs
) / 1e9

sites = SITES.split(",")
n_all = sum(
    len([f for f in os.listdir(os.path.join(TRAIN_ROOT, s)) if f.endswith((".tar.gz", ".tgz", ".tar"))])
    for s in sites
)
n_planned = min(MAX_ARCHIVES_PER_SITE * len(sites), n_all) if MAX_ARCHIVES_PER_SITE else n_all
per_date_files, per_date_gb = n_files / len(sites), size_gb / len(sites)

print(f"smoke: {n_files} files, {size_gb:.1f} GB over {len(sites)} dates")
print(f"planned ({n_planned} of {n_all} dates): ~{per_date_files * n_planned:,.0f} files, "
      f"~{per_date_gb * n_planned:.1f} GB  <- must fit BOTH local disk and Drive")
print(f"everything ({n_all} dates): ~{per_date_files * n_all:,.0f} files, ~{per_date_gb * n_all:.1f} GB")

## 4. Full conversion + scoring, cached to Drive

`--cache-dir` makes this restartable: each finished date is tarred to `MyDrive/vmax_workdir/cache_91f/<site>/<date>.tar`, and a re-run untars it instead of reconverting. When the session dies, re-run sections 1-2 and then this cell unchanged.

Two things to watch:

- **`--windows 100`**: emits one 91-step window per source file instead of three, so the output is a third the size. Colab's local disk is ~112GB and the training pipeline needs the files locally, which the 3-window conversion does not fit; with one window the whole hanam+jeju set does. The dropped windows are other time slices of the *same* scene, so this costs far less than a third of the information. Remove the flag only if you have already capped the run enough to fit.
- **`--min-free-gb 15`**: stops the loop cleanly before an archive that would fill the disk (combined CSV still written, nothing half-done left behind) instead of dying on ENOSPC. Whatever converted so far is usable - just continue to step 5.
- **`--max-archives-per-site 6`**: 6 dates per site (of hanam's 31 and jeju's 14), spread evenly over each site's date range rather than taken from the front - a contiguous run of weeks is one season and one set of construction zones, which is what a policy overfits to. Start here: it reaches training in hours instead of a day, and raising the number later only converts the newly added dates.
- **Do not mix window settings**: dates already converted with 3 windows are skipped, not re-converted, so switching mid-way leaves some dates weighted 3x. To change it, wipe `/content/data/train_91f` and the Drive `cache_91f/` + `scores/` first.
- **Drive quota**: the cache holds the whole converted dataset (the estimate printed above).
- **Wall clock**: conversion is CPU-bound and Colab gives you 2-12 vCPUs. If the extrapolation says more hours than a session allows, that is fine (the cache resumes), but a capped run gets you to training sooner.

In [ ]:
!rm -rf /content/data/smoke_91f
!uv run python scripts/prepare_archives_91f.py "$TRAIN_ROOT" /content/data/train_91f \
    --sites "$SITES" \
    --scores-dir "$DRIVE_WORKDIR/scores" \
    --cache-dir "$DRIVE_WORKDIR/cache_91f" \
    --windows 100 \
    --min-free-gb 15 \
    --max-archives-per-site "$MAX_ARCHIVES_PER_SITE"
# Raise --max-archives-per-site (or drop it for everything) once a full run has gone
# through; already-converted dates come back from the Drive cache instead of reconverting.

## 5. Build the 4 training pools

`--drop-frac 0.2` discards the bottom 20% of each site outright (near-static lane keeping - the "직진 위주 단순 데이터 제거" step); `--hard-frac 0.4` then splits what remains into hard/easy **per site**, so `hanam_*` and `jeju_*` pools stay separately weightable.

`--holdout-frac 0.02` reserves 2% of each site as `<site>_val`, excluded from every training pool. That is what section 7 validates on: BC stops when *that* loss stops improving, which it cannot detect from the training loss alone (which keeps falling while the policy is memorising expert noise). The two per-site holdouts are merged into one `val` pool because `path_dataset_val` takes a single path.

In [ ]:
!rm -rf /content/data/shards/mixture_pools /content/data/shards/val
!uv run python scripts/split_hard_easy_pools.py \
    "$DRIVE_WORKDIR/scores/combined_scores.csv" \
    /content/data/train_91f \
    /content/data/shards/mixture_pools \
    --sites "$SITES" --drop-frac 0.2 --hard-frac 0.4 --holdout-frac 0.02

# One validation pool out of the per-site holdouts (path_dataset_val takes a single path).
!uv run python scripts/merge_pools.py /content/data/shards \
    val=/content/data/shards/mixture_pools/hanam_val,/content/data/shards/mixture_pools/jeju_val

## 6. Wire up checkpoints (Drive, persistent) and the fixed eval set

In [ ]:
import os
os.makedirs(f"{DRIVE_WORKDIR}/runs", exist_ok=True)
!rm -rf /content/as-fast-as-anyone/V-Max/runs
!ln -s "$DRIVE_WORKDIR/runs" /content/as-fast-as-anyone/V-Max/runs

if HAS_EVAL_TAR:
    !mkdir -p /content/data/eval/val_sample_shards_hanam
    !tar -xf "$EVAL_TAR" -C /content/data/eval/val_sample_shards_hanam
else:
    print("no eval tar - skipping (section 9 needs it)")

## 7. Train

Weights below bias toward the evaluated site (hanam) and toward hard scenarios: hanam 0.6 / jeju 0.4, hard 0.6 / easy 0.4. Nothing is dropped inside a pool - the weight only sets how often each pool is drawn from - so this is safe to retune between runs.

`total_timesteps=5_000_000` is ~62.5k episodes at 80 steps, so with `--windows 100` (one scenario per source file) it is a few passes over the pool. Bump it (e.g. `20_000_000`, the framework default scale) once TensorBoard shows `train/imitation_loss` still trending down at 5M.

**Early stopping**: every `val_freq=100` iterations the current params are scored on the held-out `val` pool (same loss, same expert-driven unroll as training - only the scenarios differ). Each improvement rewrites `runs/<name_run>/model/model_best.pkl`; after `early_stop_patience=10` validations with no improvement the run stops and writes `early_stopped.txt`, which also makes a re-launch of the same `name_run` a no-op instead of training past the plateau. **Submit `model_best.pkl`, not `model_final.pkl`.** To only watch the curves without stopping, set `algorithm.early_stop_patience=0`; to disable validation entirely, `algorithm.val_freq=0`.

`early_stop_warmup_steps=1_000_000` is not optional. A tanh-output policy starts near zero and so do most expert actions, so the validation loss is *already low* at init and normally rises before it falls - without the warmup the run stops inside that transient and keeps the untrained network as its "best" (observed: stopped at 160k of 5M steps with best @ step 320). During the warmup the loss is logged but nothing is tracked or counted; the baseline is taken fresh at the first validation after it.

Watch these together in TensorBoard:

| curve | reading |
|---|---|
| `val/loss_vs_zero_policy` | **the one that says whether BC works at all.** 1.0 = no better than a policy that always outputs zero; expert actions cluster near zero, so the raw loss is small even untrained |
| `train/imitation_loss` vs `val/imitation_loss` | both falling = keep going; train falling while val flattens = the plateau early stopping is for |
| `val/imitation_loss_smoothed` | what patience actually judges - the raw value swings ~25% between validations from the params oscillating alone |

`batch_size=256` (up from the config's 64) is there to damp that oscillation; if the GPU runs out of memory, drop it back.

`policy.layer_sizes=[256,256]` overrides bc.yaml's `[256,64,32]` to match **sac.yaml**. This policy warm-starts SAC in section 10 and only same-shape arrays transfer, so training it at SAC's shape is what lets the hidden layers carry over and not just the encoder. It is also the shape `submission_hanam_run1/network.py` builds, so exported weights load there unchanged.

**Memory**: 4 pools means 4 live tf.data pipelines. If the session OOMs, drop `algorithm.buffer_size` to 10000 first, then `num_envs` to 2 - or merge back to 2 pools with `scripts/merge_pools.py`.

**If the session disconnects mid-run**: re-run sections 1-2, then 4 (restores from the Drive cache), 5 and 6, then this cell unchanged - `algorithm.resume=true` (default) picks up from `runs/<name_run>/model/train_state_latest.pkl` on Drive.

In [ ]:
import os

POOL_WEIGHTS = {"hanam_hard": 0.35, "hanam_easy": 0.25, "jeju_hard": 0.25, "jeju_easy": 0.15}
POOL_ROOT = "/content/data/shards/mixture_pools"

entries = []
for name, weight in POOL_WEIGHTS.items():
    pool_dir = os.path.join(POOL_ROOT, name)
    with open(os.path.join(pool_dir, "manifest.csv")) as fh:
        n = sum(1 for _ in fh) - 1  # minus header
    print(f"{name}: {n} shards, weight {weight}")
    entries.append(f"{{path: {pool_dir}/{name}.tfrecord@{n}, weight: {weight}}}")

MIXTURE = "[" + ", ".join(entries) + "]"
os.environ["MIXTURE"] = MIXTURE  # so the shell cell below sees it either way
print("\n" + MIXTURE)

with open("/content/data/shards/val/manifest.csv") as fh:
    n_val = sum(1 for _ in fh) - 1
VAL_PATH = f"/content/data/shards/val/val.tfrecord@{n_val}"
os.environ["VAL_PATH"] = VAL_PATH
print("validation:", VAL_PATH)

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
!uv run python vmax/scripts/training/train.py \
  algorithm=bc network/encoder=lq \
  algorithm.network.policy.layer_sizes=[256,256] \
  total_timesteps=5_000_000 num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=20000 \
  waymo_dataset=true \
  "mixture_datasets=$MIXTURE" \
  "path_dataset_val=$VAL_PATH" num_scenario_per_val=64 \
  algorithm.val_freq=100 algorithm.early_stop_patience=10 algorithm.early_stop_warmup_steps=1_000_000 \
  algorithm.batch_size=256 \
  name_run=colab_bc_run1 log_freq=50 save_freq=1500

## 8. Watch training in TensorBoard (optional, run in a separate cell while training runs)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/as-fast-as-anyone/V-Max/runs

## 9. After training: sweep checkpoints on held-out scenarios

This is the number that decides things, not the loss: each checkpoint drives the scenarios itself and is scored on offroad / collision / progress, the way the contest scores. `rideflux_aggregate_score` is the column closest to the contest's own.

Without the uploaded 300-scenario tar this falls back to the `val` pool from section 5 - the same scenarios early stopping selected `model_best.pkl` on, so its score here is mildly optimistic relative to the others. Scores from different sets are never comparable with each other, so keep every run on one of them.

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
import os

# The fixed 300-scenario set if it was uploaded (comparable with the local runs),
# otherwise the held-out pool from section 5.
EVAL_PATH = (
    "/content/data/eval/val_sample_shards_hanam/val_sample_shards_hanam.tfrecord@300"
    if HAS_EVAL_TAR
    else VAL_PATH
)
os.environ["EVAL_PATH"] = EVAL_PATH
os.environ["EVAL_RUN"] = "colab_bc_run2"
print("scoring against:", EVAL_PATH)

# Sweeps every model_*.pkl, so it also scores model_best.pkl (the early-stopping pick).
!uv run python scripts/evaluate_checkpoints.py \
  --name_run "$EVAL_RUN" \
  --path_dataset "$EVAL_PATH" \
  --waymo_dataset true --batch_size 4

## 10. RL fine-tuning: SAC warm-started from the BC policy

BC only ever sees expert-driven states, so it never learns to recover from its own mistakes - that is what RL is for, and starting RL from the BC weights skips the phase where a random policy crashes its way to a first reward.

**The transfer is partial, by construction.** SAC's policy emits the parameters of a distribution over actions (size 4: mean and std) where BC's emits the action itself (size 2), and SAC additionally has value networks BC never had. `pretrained_params_path` therefore grafts every array whose path *and* shape match and leaves the rest at its fresh init - with section 7's `[256,256]` and the same encoder, that is everything except the head and the value networks.

The run prints what actually transferred - `grafted N arrays, kept M at fresh init`. If N is 0 the configs do not line up; check the BC run used the same `network/encoder` and `policy.layer_sizes`. A BC run trained at bc.yaml's default `[256,64,32]` still transfers its encoder here (the bulk of the parameters), just not the hidden layers.

Point it at `model_best.pkl` (the validation-selected checkpoint), not `model_final.pkl` - on the first BC run those scored 0.585 and 0.414 on the held-out pool.

`learning_start=2000` matches the earlier local SAC runs: taking thousands of updates off a near-empty buffer is the fastest way to undo the pre-training. `total_timesteps=1_000_000` is a place to stop and compare against the BC score, not a target - raise it and re-run the same `name_run` to continue.

In [ ]:
%cd /content/as-fast-as-anyone/V-Max
BC_RUN = "colab_bc_easy1"   # the run whose model_best.pkl you want to start from
import os
BC_WEIGHTS = f"runs/{BC_RUN}/model/model_best.pkl"
os.environ["BC_WEIGHTS"] = BC_WEIGHTS
assert os.path.exists(BC_WEIGHTS), f"missing {BC_WEIGHTS}"
print("warm start from:", BC_WEIGHTS)

In [ ]:
!uv run python vmax/scripts/training/train.py \
  algorithm=sac network/encoder=lq \
  algorithm.network.policy.layer_sizes=[256,256] \
  "algorithm.pretrained_params_path=$BC_WEIGHTS" \
  total_timesteps=1_000_000 num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=50000 algorithm.learning_start=2000 \
  waymo_dataset=true \
  "mixture_datasets=$MIXTURE" \
  name_run=colab_sac_run1 log_freq=50 save_freq=500

### Score the SAC checkpoints on the same held-out pool

Same command as section 9 with the SAC run name, so the numbers sit on the same scale as the BC ones - that comparison is the only thing that says whether RL helped.

In [ ]:
!uv run python scripts/evaluate_checkpoints.py \
  --name_run colab_sac_run1 \
  --path_dataset "$EVAL_PATH" \
  --waymo_dataset true --batch_size 4

## 11. 2 try - RL fine-tuning: SAC warm-started from the BC policy
#### decrease learning_late and adjust hard /easy dataset proportion


In [ ]:
import os

POOL_WEIGHTS = {"hanam_hard": 0.5, "hanam_easy": 0.1, "jeju_hard": 0.3, "jeju_easy": 0.1}
POOL_ROOT = "/content/data/shards/mixture_pools"

entries = []
for name, weight in POOL_WEIGHTS.items():
    pool_dir = os.path.join(POOL_ROOT, name)
    with open(os.path.join(pool_dir, "manifest.csv")) as fh:
        n = sum(1 for _ in fh) - 1  # minus header
    print(f"{name}: {n} shards, weight {weight}")
    entries.append(f"{{path: {pool_dir}/{name}.tfrecord@{n}, weight: {weight}}}")

MIXTURE = "[" + ", ".join(entries) + "]"
os.environ["MIXTURE"] = MIXTURE  # so the shell cell below sees it either way
print("\n" + MIXTURE)

with open("/content/data/shards/val/manifest.csv") as fh:
    n_val = sum(1 for _ in fh) - 1
VAL_PATH = f"/content/data/shards/val/val.tfrecord@{n_val}"
os.environ["VAL_PATH"] = VAL_PATH
print("validation:", VAL_PATH)


In [ ]:
# 1. 파인튜닝 기준 체크포인트 경로 지정 (가장 성적 높았던 1.44M 체크포인트)
BEST_CHECKPOINT="/content/vmax/checkpoints/colab_sac_easy1/model_1440320.pkl"

# 2. 파인튜닝 실행
!uv run python vmax/scripts/training/train.py \
  algorithm=sac network/encoder=lq \
  algorithm.network.policy.layer_sizes=[256,256] \
  "algorithm.pretrained_params_path=$BEST_CHECKPOINT" \
  algorithm.actor_lr=6e-5 \
  algorithm.critic_lr=6e-5 \
  algorithm.target_entropy_scale=0.2 \
  total_timesteps=150_000 \
  num_envs=4 num_episode_per_epoch=1 \
  algorithm.buffer_size=50000 algorithm.learning_start=0 \
  waymo_dataset=true \
  "mixture_datasets=$MIXTURE" \
  name_run=colab_sac_finetune_v1 log_freq=20 save_freq=100